# Llama-3.2-1B × opc-sft-stage2 leaderboard — cross-model robustness cell (r=64, 9000 steps)

Cross-model cell from `~/.claude/plans/as-part-of-our-tender-quilt.md`. Same opc-sft-stage2 dataset, same horizon, same optimizer arms as Phase L — but base model swapped to Llama-3.2-1B to address the 'does this work on non-OLMo bases' reviewer objection without a full Phase B/C spend.

Cell: Llama-3.2-1B × opc-sft-stage2 (all 4 sub-configs, Llama-tokenized cache `data/opc_sft_stage2_all_packed_seq2048_llama32`) × global_batch=16 (batch=4 × accum=4) × packed_v1.1 × constant LR × α=r × all-linear × bf16 × compile × single-GPU Blackwell. `max_steps=9000`, `eval_every=250`.

Source label: **Llama OPC eval JSONL logs**: `meta-llama/Llama-3.2-1B`; `data/opc_sft_stage2_all_packed_seq2048_llama32`; `packed_v1.1`; seq2048; 9k-step eval logs.

Diagnostic q_agree source: **Llama OPC JSONL q_agree**: `optim_step.awc_q_agree_*` from `logs/chord_tight_slack_llama32_1b_opc_r256_lr2_blackwell` (r256, ns=5, lr={3e-3,1e-2}) and `logs/chord_tight_slack_llama32_1b_opc_r256_ns8_lr2_blackwell` (r256, ns=8, lr={3e-3,1e-2}). These are JSONL diagnostics, not snapshot-registry rows.

- **AdamW**: η ∈ {3e-5, 1e-4, 3e-4}
- **chord-tight k=1** (`adam-polar-product-lora-coupled-spectral-chord-tight`): η ∈ {3e-3, 1e-2, 3e-2}

Source log groups: `{adamw,chord_tight}_robustness_llama32_1b_opc_r64_blackwell` (2 groups).

**Pass criterion** (per plan): Δ direction matches OLMo (tight-chord < AdamW) at any of the 3 LRs. Magnitude can differ; direction cannot. If direction inverts, the result is OLMo-specific and must be reported as such.

**σ anchor**: borrowed `σ_AdamW(packed_v1, opc-sft-stage2, OLMo, r=64) = 0.0017` until Llama multi-seed exists. Treat σ-units as a rough scale only.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lora_playground.loader import load_runs
from lora_playground.plotting import compare_variants_figure, canonical_label
from IPython.display import display

# Single source of truth: canonical_label gives ns/picard/damping-explicit names
# (identical across every notebook); canonical colors put AdamW black + first;
# compare_variants_figure's assert_label_discriminates guard hard-errors on any
# silent merge. One render helper for every cell below.

def _key(cfg):
    return (cfg['optimizer'], float(cfg['lr']), cfg.get('muon_ns_steps'),
            cfg.get('_derived', {}).get('effective_picard_iters', cfg.get('picard_iters_override')))

def render(groups, rank, suptitle, figsize=(11, 4)):
    runs = load_runs(where={'log_group': groups}, logs_root='../logs',
                     warn_cross_commit=False, quiet=True)
    dedup = {}
    for cfg, hist in runs:
        if cfg.get('lora_r') != rank:
            continue
        k = _key(cfg)
        if k not in dedup or len(hist) > len(dedup[k][1]):
            dedup[k] = (cfg, hist)
    labeled = [(c, h) for c, h in dedup.values() if canonical_label(c) is not None]
    labels = {canonical_label(c) for c, _ in labeled}
    fig, tdf, sdf = compare_variants_figure(
        variants={l: {} for l in labels}, common_where={}, ref_label='AdamW',
        target_label='AdamW', sigma_ref=0.0017, suptitle=suptitle, figsize=figsize,
        max_steps=9000, allow_partial=True,
        prefetched_runs=labeled, variant_key=canonical_label)
    plt.show()
    display(tdf.style.format('{:.4f}', na_rep='—'))
    display(sdf.style.format({'final': '{:.4f}', 'delta': '{:+.4f}',
                              'delta_sigma': '{:+.2f}σ', 'best_lr': '{:.0e}'}, na_rep='—'))
    return tdf, sdf

## r=64

In [ ]:
GROUPS_R64 = [
    'adamw_robustness_llama32_1b_opc_r64_blackwell',
    'chord_tight_robustness_llama32_1b_opc_r64_blackwell',
    'chord_tight_robustness_llama32_1b_opc_r64_extend_left_blackwell',  # chord-tight lr-extension to {3e-4, 1e-3}
]
table_df, summary_df = render(GROUPS_R64, 64, 'Llama-3.2-1B × opc r=64')

## r=256 (rank-extension robustness)

Same two arms, r=256. Source groups: `{adamw,chord_tight}_robustness_llama32_1b_opc_r256_blackwell`.

In [ ]:
GROUPS_R256 = [
    'adamw_robustness_llama32_1b_opc_r256_blackwell',
    'adamw_robustness_llama32_1b_opc_r256_extend_left_blackwell',       # AdamW lr-extension to {3e-6, 1e-5}
    'chord_tight_robustness_llama32_1b_opc_r256_blackwell',
]
table_df_r256, summary_df_r256 = render(GROUPS_R256, 256, 'Llama-3.2-1B × opc r=256')

## NS-iteration (whitening-fraction) sweep \u2014 does Llama's optimal whitening differ from OLMo's NS-5?

chord-tight with `muon_ns_steps` \u2208 {3, 5, 8} \u2192 whitening fraction \u2248 {0.47, 0.72, 1.0} (measured on the Llama gradient spectrum, stable_rank 10.2). j=5 is the base group. AdamW shown for reference. If a non-5 j beats ns=5, Llama's optimal whitening differs from the OLMo-tuned default.

In [ ]:
GROUPS_NS = [
    'adamw_robustness_llama32_1b_opc_r64_blackwell',
    'chord_tight_robustness_llama32_1b_opc_r64_blackwell',              # ns=5 (base)
    'chord_tight_robustness_llama32_1b_opc_r64_extend_left_blackwell',  # ns=5 (lr extension)
    'chord_tight_robustness_llama32_1b_opc_r64_ns3_blackwell',          # ns=3
    'chord_tight_robustness_llama32_1b_opc_r64_ns8_blackwell',          # ns=8
]
table_df_ns, summary_df_ns = render(GROUPS_NS, 64, 'Llama-3.2-1B × opc r=64 — NS-iteration (whitening) sweep')

## Picard ablation \u2014 does the 1/\u03b7 cross-coupling add value on Llama?

chord-tight-CLEAN, polar_method=ns, muon_ns_steps=8 (full whitening). picard=1 collapses to plain polar (cross-coupling term identically zero); picard=2 activates the 1/\u03b7 coupling. If picard=2 beats picard=1 at best-lr, the coupling adds value on Llama (as it did on OLMo packed_v1). AdamW shown for reference.

In [ ]:
GROUPS_PIC = [
    'adamw_robustness_llama32_1b_opc_r64_blackwell',
    'chord_tight_robustness_llama32_1b_opc_r64_ns8_blackwell',                              # non-clean ns=8 = picard=1 ref
    'chord_tight_clean_ns8_picard_ablation_llama32_1b_opc_r64_blackwell',                   # clean ns=8 picard=2
    'chord_tight_clean_ns8_picard_ablation_llama32_1b_opc_r64_extend_left_blackwell',       # clean ns=8 picard=2 lr-extension
]
# canonical_label distinguishes 'chord-tight ns=8 k=1 (abs)' (picard=1) vs
# 'chord-tight-clean ns=8 k=2 (abs)' (picard=2) — the ablation contrast.
table_df_pic, summary_df_pic = render(GROUPS_PIC, 64, 'Llama-3.2-1B × opc r=64 — picard ablation (1/η cross-coupling)')

## r=256 — NS-iteration (whitening-fraction) sweep

chord-tight at r=256 with `muon_ns_steps` ∈ {5, 8}. ns=5 is the base `chord-tight k=1` group; ns=8 (`chord_tight_robustness_llama32_1b_opc_r256_ns8_blackwell`) is the full-whitening rank-extension. AdamW shown for reference. (No ns=3 arm at r=256.)

Diagnostic source note: `q_agree`, `chord_slack`, stable-rank, saturation, and conditioning values discussed for Llama come from JSONL `optim_step` events in `logs/chord_tight_slack_llama32_1b_opc_r256_lr2_blackwell` and `logs/chord_tight_slack_llama32_1b_opc_r256_ns8_lr2_blackwell`. Those are Llama OPC packed-v1.1 runs using `data/opc_sft_stage2_all_packed_seq2048_llama32`; they are not snapshot-registry rows.


In [ ]:
GROUPS_NS_R256 = [
    'adamw_robustness_llama32_1b_opc_r256_blackwell',
    'adamw_robustness_llama32_1b_opc_r256_extend_left_blackwell',
    'chord_tight_robustness_llama32_1b_opc_r256_blackwell',         # ns=5 (base)
    'chord_tight_robustness_llama32_1b_opc_r256_ns8_blackwell',     # ns=8
]
table_df_ns_r256, summary_df_ns_r256 = render(GROUPS_NS_R256, 256, 'Llama-3.2-1B × opc r=256 — NS-iteration (whitening) sweep')

## r=256 — Picard ablation (does the 1/η cross-coupling add value?)

Same picard ablation as r=64, at r=256. `ns8 picard=1 (ref, non-clean)` = `chord_tight_robustness_llama32_1b_opc_r256_ns8_blackwell` (cross-coupling identically zero); `clean ns8 picard=2` = `chord_tight_clean_ns8_picard_ablation_llama32_1b_opc_r256_blackwell` (1/η coupling active). Both arms share lr ∈ {3e-4, 1e-3, 3e-3, 1e-2}. AdamW shown for reference.

In [ ]:
GROUPS_PIC_R256 = [
    'adamw_robustness_llama32_1b_opc_r256_blackwell',
    'adamw_robustness_llama32_1b_opc_r256_extend_left_blackwell',
    'chord_tight_robustness_llama32_1b_opc_r256_ns8_blackwell',                  # non-clean ns=8 = picard=1 ref
    'chord_tight_clean_ns8_picard_ablation_llama32_1b_opc_r256_blackwell',       # clean ns=8 picard=2
]
table_df_pic_r256, summary_df_pic_r256 = render(GROUPS_PIC_R256, 256, 'Llama-3.2-1B × opc r=256 — picard ablation (1/η cross-coupling)')